**Vanguard**  
**Jaydon Hylton (2210144)** , **Chadwick Cox (1800729)** , **Semoy Smith (1505625)**

# Part 3:  Structured Input/Output Validation with Pydantic

# Vanguard_part3_structured_io

**Group:** Vanguard  
**Jaydon Hylton (2210144)** **Chadwick Cox (1800729)** **Semoy Smith (1505625)**

`AcademicAuditReport` with `Field(description=...)`. Bound via `.with_structured_output(..., method="json_schema")`.

In [ ]:
import json
import re
from typing import List

from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator

from vanguard.azure_chat import build_chat_model

# Typical free-text reply: readable, but it both clears the student and says an elective is still missing.
raw_llm_text = """
Sure! Here is the evaluation for student ID 816001234: The student has a GPA of 3.65.
Their academic standing is Honors. They are cleared for graduation, but need 1 elective in CS.
""".strip()
print(raw_llm_text)


# Field descriptions go to the model so each JSON key has a meaning, not just a type.
class AcademicAuditReport(BaseModel):
    student_id: str = Field(description="Unique 9-digit student identification number")
    gpa: float = Field(description="Cumulative Grade Point Average (0.00 to 4.00)")
    academic_standing: str = Field(
        description="Status: Good Standing, Probation, or Honors"
    )
    graduation_cleared: bool = Field(
        description="True only when pending_requirements is empty; otherwise false"
    )
    pending_requirements: List[str] = Field(
        description="Remaining required electives; non-empty means not cleared"
    )

    # IDs in this dataset are always nine digits; anything else is a bad extract.
    @field_validator("student_id")
    @classmethod
    def nine_digit_id(cls, value: str) -> str:
        value = value.strip()
        if not re.fullmatch(r"\d{9}", value):
            raise ValueError("student_id must be exactly nine digits")
        return value

    # Cannot mark graduation cleared while anything is still on pending_requirements.
    @model_validator(mode="after")
    def pending_blocks_clearance(self) -> "AcademicAuditReport":
        if self.pending_requirements and self.graduation_cleared:
            raise ValueError("graduation_cleared cannot be true while requirements are pending")
        return self


# Build a valid report by hand so we can see the JSON shape before asking the model for it.
validated_output = AcademicAuditReport(
    student_id="816001234",
    gpa=3.65,
    academic_standing="Honors",
    graduation_cleared=False,
    pending_requirements=["CS Elective Level 3"],
)
print(json.dumps(validated_output.model_dump(), indent=2))

# This combination should fail: cleared=True while an elective is still pending.
try:
    AcademicAuditReport(
        student_id="816001234",
        gpa=3.65,
        academic_standing="Honors",
        graduation_cleared=True,
        pending_requirements=["CS Elective Level 3"],
    )
except ValidationError as error:
    print(error.errors()[0]["msg"])

In [ ]:
# Ask the model to fill AcademicAuditReport from the messy paragraph above.
# Remaining electives should force graduation_cleared to false.
report = (
    build_chat_model("local")
    .with_structured_output(AcademicAuditReport, method="json_schema")
    .invoke(
        raw_llm_text
        + "\n\nExtract AcademicAuditReport. "
        "The source text is contradictory (cleared vs remaining elective). "
        "If any elective remains, set graduation_cleared to false "
        "and put that elective in pending_requirements."
    )
)
print(json.dumps(report.model_dump(), indent=2))